#  C++ Primer : Multi-dimensional Arrays
# 1. Declaring and using
```
double a[3][4];        // 3 rows, 4 columns
a[i][j] = 1.0;         // row i, column j
```

* `a[3][4]` is literally "an array of 3 things, each of which is an array of 4 `double`s"
* It is stored in _row-major_ order: all of row 0, then all of row 1, then all of row 2, contiguously in memory
```
a[0][0] a[0][1] a[0][2] a[0][3] a[1][0] a[1][1] ... a[2][3]
```
* Therefore `a[i][j]` sits at offset `i * 4 + j` from the beginning. The compiler needs the number of _columns_ to compute that offset, which has a consequence:

# 2. Passing one to a function
* You may omit the first dimension but not the others
```
double sum(double a[][4], long m);   // ok: columns known
double sum(double a[][], long m, long n);  // not allowed
```
* That is inconvenient when the size is decided at run time, so parallel programs usually do the index arithmetic by hand on a flat, one-dimensional array
```
double sum(double * a, long m, long n) {   /* a has m * n elements */
  ... a[i * n + j] ...
}
```
* You will see both styles in this course. They describe exactly the same memory


In [ ]:
%%writefile rowmajor.cc
#include <stdio.h>

/* a is m x n, stored flat in row-major order */
void fill(double * a, long m, long n) {
  for (long i = 0; i < m; i++) {
    for (long j = 0; j < n; j++) {
      a[i * n + j] = 10 * i + j;
    }
  }
}

int main() {
  const long m = 3, n = 4;
  double a[m][n];
  fill(&a[0][0], m, n);     /* pass the address of the first element */

  for (long i = 0; i < m; i++) {
    for (long j = 0; j < n; j++) {
      printf("a[%ld][%ld] = %4.1f  at %p\n", i, j, a[i][j], (void *)&a[i][j]);
    }
  }
  return 0;
}

* `&x` is "the address of `x`" and `%p` prints an address. We use them here only to look at the layout; addresses are the subject of a later notebook
* Note that `fill` wrote through a flat `double *` and `main` read back through `a[i][j]`, and they agree


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall rowmajor.cc -o rowmajor

In [ ]:
%%bash
./rowmajor

* <font color="green">Exercise: look at the printed addresses. By how many bytes do consecutive elements of a row differ? And consecutive elements of a column (`a[0][0]`, `a[1][0]`, `a[2][0]`)? Explain both numbers. Then swap the two loops in `fill` so that `j` is the outer loop, and check that the output is unchanged --- why is it unchanged, and what did change?</font>

* Start from this copy, so the program above stays around to compare with


In [ ]:
%%writefile rowmajor_ex.cc
#include <stdio.h>

void fill(double * a, long m, long n) {
  for (long i = 0; i < m; i++) {           /* try swapping the two loops */
    for (long j = 0; j < n; j++) {
      a[i * n + j] = 10 * i + j;
    }
  }
}

int main() {
  const long m = 3, n = 4;
  double a[m][n];
  fill(&a[0][0], m, n);
  for (long i = 0; i < m; i++) {
    for (long j = 0; j < n; j++) {
      printf("a[%ld][%ld] = %4.1f  at %p\n", i, j, a[i][j], (void *)&a[i][j]);
    }
  }
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall rowmajor_ex.cc -o rowmajor_ex

In [ ]:
%%bash
./rowmajor_ex